In [7]:
import os, time, warnings
import numpy as np, pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
warnings.filterwarnings("ignore")

def pick(*c): return next(p for p in c if os.path.exists(p))
DATA = pick("../../exps_/data/feature1", "/mnt/user-data/uploads")
RAW  = pick("../../data", "/mnt/user-data/uploads")
RES  = "../../exps_/results" if os.path.exists("../../exps_/results") else "/mnt/user-data/outputs/prj/exps_results"
os.makedirs(RES, exist_ok=True)

x_train = pd.read_excel(f"{DATA}/x_train.xlsx").fillna(0)
x_test  = pd.read_excel(f"{DATA}/x_test.xlsx").fillna(0)
y_train = pd.read_excel(f"{DATA}/y_train.xlsx")["SalePrice"]
X_tr, X_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)
rmse = lambda y, p: float(np.sqrt(mean_squared_error(y, p)))
cv3 = KFold(3, shuffle=True, random_state=42)
rows = []
def log(name, model, note=""):
    p = model.predict(X_val)
    rows.append(dict(stage="5_tuned", model=name, features=X_tr.shape[1], val_rmse=rmse(y_val, p),
                     val_mae=float(mean_absolute_error(y_val, p)), val_r2=float(r2_score(y_val, p)), note=note))
    print(f"{name:<26} Val RMSE={rows[-1]['val_rmse']:.4f}  {note}")

In [8]:
# 1) GridSearchCV cho mo hinh tuyen tinh (CV 5-fold tren tap train 80%)
cv5 = KFold(5, shuffle=True, random_state=42)
g_ridge = GridSearchCV(Ridge(), {"alpha": [0.1, 0.5, 1, 3, 10, 30, 100, 300]}, cv=cv5, scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)
g_lasso = GridSearchCV(Lasso(max_iter=50000), {"alpha": [1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2]}, cv=cv5, scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)
g_enet  = GridSearchCV(ElasticNet(max_iter=50000), {"alpha": [2e-4, 5e-4, 1e-3, 2e-3], "l1_ratio": [0.2, 0.5, 0.8]}, cv=cv5, scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)
for n, g in [("Ridge", g_ridge), ("Lasso", g_lasso), ("ElasticNet", g_enet)]:
    print(f"{n}: best={g.best_params_}  CV RMSE={-g.best_score_:.4f}")
    log(n + " (tuned)", g.best_estimator_, str(g.best_params_))

Ridge: best={'alpha': 300}  CV RMSE=0.1435
Ridge (tuned)              Val RMSE=0.1245  {'alpha': 300}
Lasso: best={'alpha': 0.005}  CV RMSE=0.1481
Lasso (tuned)              Val RMSE=0.1252  {'alpha': 0.005}
ElasticNet: best={'alpha': 0.002, 'l1_ratio': 0.8}  CV RMSE=0.1494
ElasticNet (tuned)         Val RMSE=0.1193  {'alpha': 0.002, 'l1_ratio': 0.8}


In [9]:
# 2) RandomizedSearchCV cho mo hinh Boosting (CV 3-fold)
rs_gbr = RandomizedSearchCV(GradientBoostingRegressor(random_state=42, subsample=0.8),
    {"n_estimators": randint(300, 900), "learning_rate": uniform(0.01, 0.05), "max_depth": randint(2, 5),
     "min_samples_leaf": randint(3, 15), "max_features": uniform(0.2, 0.5)},
    n_iter=10, cv=cv3, scoring="neg_root_mean_squared_error", random_state=42).fit(X_tr, y_tr)
print("GBR best:", {k: (round(float(v), 4)) for k, v in rs_gbr.best_params_.items()}, " CV RMSE=%.4f" % -rs_gbr.best_score_)
log("Gradient Boosting (tuned)", rs_gbr.best_estimator_)

rs_hgb = RandomizedSearchCV(HistGradientBoostingRegressor(random_state=42),
    {"max_iter": randint(300, 900), "learning_rate": uniform(0.01, 0.05), "max_depth": randint(2, 6),
     "min_samples_leaf": randint(5, 30), "l2_regularization": uniform(0, 5)},
    n_iter=15, cv=cv3, scoring="neg_root_mean_squared_error", random_state=42).fit(X_tr, y_tr)
print("HGB best:", {k: (round(float(v), 4)) for k, v in rs_hgb.best_params_.items()}, " CV RMSE=%.4f" % -rs_hgb.best_score_)
log("HistGB (tuned)", rs_hgb.best_estimator_)

GBR best: {'learning_rate': 0.0362, 'max_depth': 4.0, 'max_features': 0.2697, 'min_samples_leaf': 14.0, 'n_estimators': 489.0}  CV RMSE=0.1267
Gradient Boosting (tuned)  Val RMSE=0.1358  
HGB best: {'l2_regularization': 2.8522, 'learning_rate': 0.036, 'max_depth': 3.0, 'max_iter': 595.0, 'min_samples_leaf': 25.0}  CV RMSE=0.1315
HistGB (tuned)             Val RMSE=0.1406  


In [10]:
# 3) Ensemble (da tune) - moi mo hinh tuyen tinh + Gradient Boosting
class Avg:
    def __init__(self, ms, w): self.ms, self.w = ms, np.array(w)
    def fit(self, X, y):
        for m in self.ms: m.fit(X, y)
        return self
    def predict(self, X): return np.column_stack([m.predict(X) for m in self.ms]) @ self.w

enet_b, ridge_b, lasso_b, gbr_b, hgb_b = (g_enet.best_estimator_, g_ridge.best_estimator_, g_lasso.best_estimator_,
                                          rs_gbr.best_estimator_, rs_hgb.best_estimator_)
from sklearn.base import clone
cands = {
    "Ridge (tuned)":              lambda: clone(ridge_b),
    "ElasticNet (tuned)":         lambda: clone(enet_b),
    "Gradient Boosting (tuned)":  lambda: clone(gbr_b),
    "HistGB (tuned)":             lambda: clone(hgb_b),
    "Ensemble 0.5 ENet + 0.5 GBR": lambda: Avg([clone(enet_b), clone(gbr_b)], [0.5, 0.5]),
    "Ensemble ENet/Ridge/GBR/HGB": lambda: Avg([clone(enet_b), clone(ridge_b), clone(gbr_b), clone(hgb_b)], [0.25, 0.25, 0.25, 0.25]),
}
for n in ("Ensemble 0.5 ENet + 0.5 GBR", "Ensemble ENet/Ridge/GBR/HGB"):
    log(n, cands[n]().fit(X_tr, y_tr))

Ensemble 0.5 ENet + 0.5 GBR Val RMSE=0.1227  
Ensemble ENet/Ridge/GBR/HGB Val RMSE=0.1242  


In [11]:
# 4) Chon mo hinh cuoi bang CV 5-fold tren TOAN BO train (1460 dong) - on dinh hon Val 292 dong
cvf = KFold(5, shuffle=True, random_state=7)
def cv_rmse(make):
    out = []
    for a, b in cvf.split(x_train):
        m = make().fit(x_train.iloc[a], y_train.iloc[a])
        out.append(rmse(y_train.iloc[b], m.predict(x_train.iloc[b])))
    return float(np.mean(out)), float(np.std(out))
cv_tab = []
for n, mk in cands.items():
    mu, sd = cv_rmse(mk); cv_tab.append(dict(model=n, cv_rmse=mu, cv_std=sd)); print(f"{n:<30} CV5 RMSE = {mu:.4f} +- {sd:.4f}")
cv_tab = pd.DataFrame(cv_tab).sort_values("cv_rmse"); cv_tab.to_csv(f"{RES}/sklearn_cv5.csv", index=False)
best_name = cv_tab.iloc[0]["model"]; print("\n=> Mo hinh tot nhat theo CV5:", best_name)

Ridge (tuned)                  CV5 RMSE = 0.1387 +- 0.0434
ElasticNet (tuned)             CV5 RMSE = 0.1412 +- 0.0501
Gradient Boosting (tuned)      CV5 RMSE = 0.1270 +- 0.0239
HistGB (tuned)                 CV5 RMSE = 0.1301 +- 0.0216
Ensemble 0.5 ENet + 0.5 GBR    CV5 RMSE = 0.1266 +- 0.0368
Ensemble ENet/Ridge/GBR/HGB    CV5 RMSE = 0.1259 +- 0.0347

=> Mo hinh tot nhat theo CV5: Ensemble ENet/Ridge/GBR/HGB


In [12]:
# 5) Fit lai tren TOAN BO train, du doan test, xuat submission_sklearn.csv
model = cands[best_name]().fit(x_train, y_train)
pred = np.expm1(model.predict(x_test))              # dao nguoc log1p
sub = pd.read_csv(f"{RAW}/sample_submission.csv")
assert len(sub) == len(pred) == 1459 and np.isfinite(pred).all()
sub["SalePrice"] = pred
sub.to_csv(f"{RES}/submission_sklearn.csv", index=False)
pd.DataFrame(rows).sort_values("val_rmse").to_csv(f"{RES}/sklearn_results_step5.csv", index=False)
print("Da luu submission_sklearn.csv:", sub.shape, "| min/median/max:", int(pred.min()), int(np.median(pred)), int(pred.max()))
print(sub.head())

Da luu submission_sklearn.csv: (1459, 2) | min/median/max: 45409 156420 693121
     Id      SalePrice
0  1461  119646.456080
1  1462  161950.294399
2  1463  177337.257283
3  1464  194847.098170
4  1465  192083.998997
